In [17]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [18]:
import jax
import jax.numpy as jnp
import random
import numpy as np
from einops import rearrange
from jax import jit, vmap
from hoam.plot import scatter_movie

In [19]:
omega = 14
    
def get_inital_condition(key):
    mu_0 = jnp.asarray([1, 1, 0, 0])
    ic = jax.random.normal(key, (4,))
    ic = (ic*0.1) - mu_0
    return ic

def drift(t, y, *args):

    x1, x2, x3, x4 = y
    x1_dot = x3
    x2_dot = x4
    x3_dot = -omega**2*x1
    x4_dot = -omega**2*x2
    return jnp.asarray([x1_dot, x2_dot, x3_dot, x4_dot])

def diffusion(t, y, *args):
    return jnp.asarray([0, 0, 1, 1])*5e-2

seed = 1
key = jax.random.key(seed)
dt = 5e-4
t_end = 1
t_eval = np.linspace(0.0, t_end, int(t_end/dt)+1)
   
n_samples = 10_000

In [20]:
from hoam.sde import solve_sde

sol = solve_sde(drift, diffusion, t_eval, get_inital_condition, n_samples, dt=1e-3, key=key)
sol = sol[..., :2]
sol = rearrange(sol, 'N T D -> T N D')

In [21]:
scatter_movie(sol)

In [22]:
from hoam.colora import build_colora

key = jax.random.PRNGKey(1)

x_dim = 2
mu_t_dim = 2
u_dim = 1

u_hat_config = {'width': 25, 'layers': ['C']*7}
h_config = {'width': 15, 'layers': ['D']*3}

s_fn, params_init = build_colora(u_hat_config, h_config, x_dim, mu_t_dim, u_dim, rank=3, key=key)

In [23]:
from hoam.utils import get_rand_idx, interplate_in_t
from hoam.quad import get_simpsons

# here we prepare the data for quadtrature

bs_t = 256 # batch size in time
bs_n = 256 # batch size in samples

# necessary for simpsons
if (bs_t - 1) % 2 != 0:
    bs_t += 1
    
t_batch, quad_weights = get_simpsons(bs_t)
start, end = jnp.asarray([0]), jnp.asarray([1.0])
t_batch = jnp.concatenate([start, t_batch, end])
t_batch = t_batch.reshape(-1, 1)
        
X_batch = interplate_in_t(sol, t_eval, t_batch)
mu_batch = jnp.asarray([1.0]) # dummy mu for this example
    
print(X_batch.shape, t_batch.shape)

(259, 10000, 2) (259, 1)


In [24]:

def get_sample_fn(X_batch, t_batch, mu_batch):
    def sample_fn(in_key):

        nonlocal X_batch
        nonlocal t_batch
        nonlocal mu_batch
    
        T, N, D = X_batch.shape
        T, one = t_batch.shape
        
        keys = jax.random.split(in_key, num=T)
        sample_idx = vmap(get_rand_idx, (0, None, None))(keys, N, bs_n)

        rows = jnp.arange(X_batch.shape[0])[:, jnp.newaxis]
        X_batch = X_batch[rows, sample_idx]

        return mu_batch, X_batch, t_batch, quad_weights

    return sample_fn

sample_fn = get_sample_fn(X_batch, t_batch, mu_batch)

In [25]:
from hoam.utils import meanvmap, hess_trace_estimator, tracewrap
from jax import jacrev, jacfwd

s_Ex = meanvmap(s_fn, in_axes=(None, 0, None, None))
s_Ex_Vt = vmap(s_Ex, in_axes=(None, 0, 0, None))

s_dx = jacrev(s_fn, 1)
s_dx_norm = lambda *args: jnp.sum(s_dx(*args)**2)
s_dx_norm_Ex = meanvmap(s_dx_norm, in_axes=(None, 0, None, None))
s_dx_norm_Ex_Vt = vmap(s_dx_norm_Ex, in_axes=(None, 0, 0, None))

s_dt = jacrev(s_fn, 2)
dt_Ex = meanvmap(s_dt, in_axes=(None, 0, None, None))
dt_Ex_Vt = vmap(dt_Ex, in_axes=(None, 0, 0, None))

laplace = tracewrap(jacfwd(s_dx, 1))
laplace_norm = lambda *args: jnp.sum(laplace(*args)**2)
laplace_norm_Ex = meanvmap(laplace_norm, in_axes=(None, 0, None, None))
laplace_norm_Ex_Vt = vmap(laplace_norm_Ex, in_axes=(None, 0, 0, None))


epsilon = 1e-2

def loss_fn(params, mu, X_batch, t_batch, quad_weights):
    
    
    boundary_term = s_Ex(mu, X_batch[0], t_batch[0], params) - s_Ex(mu,  X_batch[-1], t_batch[-1], params)

    X_batch = X_batch[1:-1]
    t_batch = t_batch[1:-1]
    
    grad = s_dx_norm_Ex_Vt(mu, X_batch, t_batch, params)
    dt = dt_Ex_Vt(mu, X_batch, t_batch, params)
    dt = jnp.squeeze(dt)
    
    if epsilon > 0.0:
        lap = laplace_norm_Ex_Vt(mu, X_batch, t_batch, params)
    else:
        lap = 0.0
    
    interior = 0.5*grad + dt + epsilon**2*0.5*lap

    interior_loss = (interior * quad_weights).sum()
    
    loss = interior_loss + boundary_term
    
    return loss

In [26]:
from hoam.adam import adam_opt

opt_params, loss_history = adam_opt(params_init, loss_fn, sample_fn, steps=10_000, learning_rate=2e-3, verbose=True, key=key)


  0%|          | 0/10000 [00:00<?, ?it/s]

In [14]:
from hoam.sde import solve_sde_ic

def solve_test_sde(s_fn, params, ics, t_int, dt, sigma, mu, key):
    s_dx = jacrev(s_fn, 1)

    def drift(t, y, *args):
        t = jnp.asarray([t])
        f = jnp.squeeze(s_dx(mu, y, t, params))
        return f

    def diffusion(t, y, *args):
        return epsilon * jnp.ones_like(y)

    keys = jax.random.split(key, num=len(ics))
    test_sol = vmap(solve_sde_ic, (0, 0, None, None, None, None))(
        ics, keys, t_int, dt, drift, diffusion)
    test_sol = rearrange(test_sol, 'N T D -> T N D')

    return test_sol


In [15]:
ic = sol[0]
t_int =  jnp.linspace(0.0, 1.0, len(t_eval))
dt_test = 1e-3
test_sol = solve_test_sde(s_fn, opt_params, ic, t_int, dt_test, epsilon, mu_batch, key)


In [16]:
n_plot = 512
N = test_sol.shape[1]
idx_sample = np.linspace(0, N-1, n_plot, dtype=np.uint32)
cs = [*['r']*n_plot, *['b']*n_plot]
plot_sol = jnp.hstack([sol[:, idx_sample], test_sol[:, idx_sample]])
scatter_movie(plot_sol, c=cs, alpha=0.2)